In [2]:
import numpy as np
from scipy.stats import norm
import pandas as pd

# ======================
# fixed parameters
# ======================
K = 100
#r = 0.05
N = 50000        # Monte Carlo steps
M_half = 126     # half year steps
M_full = 252     # one year steps

# dynamic parameters
S_list = [80, 90, 100, 110, 120]
sigma_list = [0.15, 0.2, 0.25, 0.3, 0.4]
T_list = [0.5, 1.0]
R_list=[0.05,0.055]
# ======================
# 1. closed-form
# ======================
def closed_european(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)

def closed_binary(S, K, T, r, sigma):
    d2 = (np.log(S/K) + (r - 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return np.exp(-r*T) * norm.cdf(d2)

# ======================
# 2. closed-form Monte Carlo
# ======================
def closed_mc(S, K, T, r, sigma, N):
    N_half = N // 2
    Z = np.random.normal(0, 1, N_half)
    ST = S * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    eu_pay = np.maximum(ST - K, 0)
    bi_pay = (ST > K).astype(float)
    df = np.exp(-r*T)
    return df*eu_pay.mean(), df*eu_pay.std()/np.sqrt(N), df*bi_pay.mean(), df*bi_pay.std()/np.sqrt(N)

# ======================
# 2. Monte Carlo antithetic
# ======================
def closed_mc_anti(S, K, T, r, sigma, N):
    N_half = N // 2
    Z = np.random.normal(0, 1, N_half)
    ST1 = S * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    ST2 = S * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*(-Z))
    ST = np.concatenate([ST1, ST2])
    
    
    eu_pay = np.maximum(ST - K, 0)
    bi_pay = (ST > K).astype(float)
    df = np.exp(-r*T)
    return df*eu_pay.mean(), df*eu_pay.std()/np.sqrt(N), df*bi_pay.mean(), df*bi_pay.std()/np.sqrt(N)

# ======================
# 3. Euler Monte Carlo
# ======================
def euler_mc(S, K, T, r, sigma, N, M):
    N_half, dt = N // 2, T/M
    ST= np.full(N_half, S, dtype=np.float64)
    for _ in range(M):
        Z = np.random.normal(0,1,N_half)
        ST *= 1 + r*dt + sigma*np.sqrt(dt)*Z
    eu_pay = np.maximum(ST-K,0)
    bi_pay = (ST>K).astype(float)
    df = np.exp(-r*T)
    return df*eu_pay.mean(), df*eu_pay.std()/np.sqrt(N), df*bi_pay.mean(), df*bi_pay.std()/np.sqrt(N)

# ======================
# 3. Euler + antithetic
# ======================
def euler_anti(S, K, T, r, sigma, N, M):
    N_half, dt = N // 2, T/M
    # 显式指定dtype=np.float64，避免整型数组冲突
    S1, S2 = np.full(N_half, S, dtype=np.float64), np.full(N_half, S, dtype=np.float64)
    for _ in range(M):
        Z = np.random.normal(0,1,N_half)
        S1 *= 1 + r*dt + sigma*np.sqrt(dt)*Z
        S2 *= 1 + r*dt + sigma*np.sqrt(dt)*(-Z)
    ST = np.concatenate([S1,S2])
    eu_pay = np.maximum(ST-K,0)
    bi_pay = (ST>K).astype(float)
    df = np.exp(-r*T)
    return df*eu_pay.mean(), df*eu_pay.std()/np.sqrt(N), df*bi_pay.mean(), df*bi_pay.std()/np.sqrt(N)

# ======================
# 4. Milstein Monte Carlo
# ======================
def milstein_mc(S, K, T, r, sigma, N, M):
    N_half, dt = N//2, T/M
    ST = np.full(N_half, S, dtype=np.float64)
    for _ in range(M):
        Z = np.random.normal(0,1,N_half)
        corr = 0.5*sigma**2 * (Z**2 -1)*dt
        ST *= 1 + r*dt + sigma*np.sqrt(dt)*Z + corr
    eu_pay = np.maximum(ST-K,0)
    bi_pay = (ST>K).astype(float)
    df = np.exp(-r*T)
    return df*eu_pay.mean(), df*eu_pay.std()/np.sqrt(N), df*bi_pay.mean(), df*bi_pay.std()/np.sqrt(N)
# ======================
# 4. Milstein + antithetic
# ======================
def milstein_anti(S, K, T, r, sigma, N, M):
    N_half, dt = N//2, T/M
    # 显式指定dtype=np.float64，避免整型数组冲突
    S1, S2 = np.full(N_half, S, dtype=np.float64), np.full(N_half, S, dtype=np.float64)
    for _ in range(M):
        Z = np.random.normal(0,1,N_half)
        corr = 0.5*sigma**2 * (Z**2 -1)*dt
        S1 *= 1 + r*dt + sigma*np.sqrt(dt)*Z + corr
        S2 *= 1 + r*dt + sigma*np.sqrt(dt)*(-Z) + corr
    ST = np.concatenate([S1,S2])
    eu_pay = np.maximum(ST-K,0)
    bi_pay = (ST>K).astype(float)
    df = np.exp(-r*T)
    return df*eu_pay.mean(), df*eu_pay.std()/np.sqrt(N), df*bi_pay.mean(), df*bi_pay.std()/np.sqrt(N)

results = []
for r in R_list:
 for T in T_list:
    M = M_half if T == 0.5 else M_full
    for S in S_list:
        for sigma in sigma_list:
            # closed-form
            eu_cf = closed_european(S,K,T,r,sigma)
            bi_cf = closed_binary(S,K,T,r,sigma)
            
            # closed form , E_M, Milstein 
            eu_cm, se_cm, bi_cm, se_bcm = closed_mc(S,K,T,r,sigma,N)
            eu_el, se_el, bi_el, se_bel = euler_mc(S,K,T,r,sigma,N,M)
            eu_mi, se_mi, bi_mi, se_bmi = milstein_mc(S,K,T,r,sigma,N,M)
            
            results.append([
                round(T,1), S, sigma,r,
                round(eu_cf,4), round(eu_cm,4), round(se_cm,5),
                round(eu_el,4), round(se_el,5), round(eu_mi,4), round(se_mi,5),
                round(bi_cf,4), round(bi_cm,4), round(se_bcm,5),
                round(bi_el,4), round(se_bel,5), round(bi_mi,4), round(se_bmi,5)
            ])

# ======================
# Create DataFrame and HTML
# ======================
df = pd.DataFrame(results, columns=[
    "T", "S", "σ","r",
    "EU_Closed", "EU_ClosedMC", "EU_CM_SE",
    "EU_Euler", "EU_Euler_SE", "EU_Milstein", "EU_Mil_SE",
    "Binary_Closed", "Binary_ClosedMC", "Binary_CM_SE",
    "Binary_Euler", "Binary_Euler_SE", "Binary_Milstein", "Binary_Mil_SE"
])

# create table
df.to_html("option_pricing_resultsmc.html", index=False, border=2, justify="center")
print("✅ Completed：option_pricing_results_mc.html")

# Anti result

results = []
for r in R_list:
 for T in T_list:
    M = M_half if T == 0.5 else M_full
    for S in S_list:
        for sigma in sigma_list:
            # closed-form
            eu_cf = closed_european(S,K,T,r,sigma)
            bi_cf = closed_binary(S,K,T,r,sigma)
        
            # closed-form, E_M, Milstein + anti
            eu_cm, se_cm, bi_cm, se_bcm = closed_mc_anti(S,K,T,r,sigma,N)
            eu_el, se_el, bi_el, se_bel = euler_anti(S,K,T,r,sigma,N,M)
            eu_mi, se_mi, bi_mi, se_bmi = milstein_anti(S,K,T,r,sigma,N,M)
            
            results.append([
                round(T,1), S, sigma,r,
                round(eu_cf,4), round(eu_cm,4), round(se_cm,5),
                round(eu_el,4), round(se_el,5), round(eu_mi,4), round(se_mi,5),
                round(bi_cf,4), round(bi_cm,4), round(se_bcm,5),
                round(bi_el,4), round(se_bel,5), round(bi_mi,4), round(se_bmi,5)
            ])

# ======================
# create DataFrame and HTML
# ======================
df = pd.DataFrame(results, columns=[
    "T", "S", "σ","r",
    "EU_Closed", "EU_Closedanti", "EU_CM_SE",
    "EU_Euleranti", "EU_Euler_SE", "EU_Milsteinanti", "EU_Mil_SE",
    "Binary_Closed", "Binary_Closedanti", "Binary_CM_SE",
    "Binary_Euleranti", "Binary_Euler_SE", "Binary_Milsteinanti", "Binary_Mil_SE"
])

# create html table
df.to_html("option_pricing_results_anti.html", index=False, border=2, justify="center")
print("✅ Completed and saved ：option_pricing_results_anti.html")

==================== Complete option price ====================
  T   S  sigma  EU_Closed  EU_ClosedMC  EU_Euler  EU_Milstein  Binary_Closed  Binary_ClosedMC  Binary_Euler  Binary_Milstein
0.5  80   0.15     0.1123       0.1132    0.1111       0.1150         0.0267           0.0273        0.0270           0.0273
0.5  80   0.20     0.4562       0.4576    0.4602       0.4512         0.0688           0.0691        0.0702           0.0690
0.5  80   0.25     1.0254       1.0304    1.0020       1.0337         0.1105           0.1129        0.1110           0.1117
0.5  80   0.30     1.7611       1.7446    1.7391       1.7435         0.1455           0.1439        0.1433           0.1446
0.5  80   0.40     3.5463       3.6654    3.5327       3.5463         0.1950           0.1961        0.1945           0.1939
0.5  90   0.15     1.2853       1.2869    1.2734       1.2799         0.2036           0.2021        0.2037           0.2052
0.5  90   0.20     2.3494       2.3280    2.3598       2.3660